# **LRP: Practice**

Hooray, everyone! You are at the final practice on gradient methods of the [Explainable AI](https://ai-interpretability.school) course. In it we will figure out Layer-wise Relevance Propagation.

In the theory we noted the fact that the nonlinearity of the transformations in a DNN requires modifications in the computation, which we are able to carry out. In particular, the LRP rules concern *deep rectiﬁer networks* — those that include layers $a_k$ of the form: $a_k=max(0, \sum_{0,j}a_jw_jk)$.

Let us recall what the basic relevance of a neuron $R_j$, received from the neurons of the next layer, looks like:

$$R_j = \sum_k\frac{a_jw_{jk}}{\sum_{0,j}a_jw_{jk}}R_k$$

What if $a_j$ or $w_j$ are equal to zero?

In this practice you will:
- Study the use of the LRP method with the [captum](https://captum.ai/) library
- Learn to apply different modifications of the LRP computation to particular layers of the network
- Compare the effect of applying the LRP rules and of not applying them, using one of the quality metrics for activation maps — the Gini index.

In [ ]:
!pip install captum -q

In [ ]:
import torch
import torch.nn.functional as F

from PIL import Image

import os
import json
import requests
import numpy as np
from io import BytesIO

from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt

import torchvision
from torchvision import models
from torchvision import transforms

from captum.attr import LRP
from captum.attr import visualization as viz
from captum.attr._utils.lrp_rules import EpsilonRule, GammaRule, Alpha1_Beta0_Rule

import urllib

url = "https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/imagenet_classes.txt"
urllib.request.urlretrieve(url, "imagenet_classes.txt")

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

Let us load the working image and the model itself. In this practice we come back to VGG16,
 since the architecture of this model is convenient for practising the basic LRP rules discussed in the module.

In [ ]:
transform = transforms.Compose([
 transforms.Resize(256),
 transforms.CenterCrop(224),
 transforms.ToTensor(),
 transforms.Normalize(mean=[0.485, 0.456, 0.406],
                      std=[0.229, 0.224, 0.225]
 )
])

url = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/cat_and_dog.jpg'

image_bytes = requests.get(url).content
image = Image.open(BytesIO(image_bytes))  # Let us load the friends we already know

plt.figure(figsize=(8,10))
plt.axis('off')
plt.imshow(image);

In [ ]:
resize_transform = transforms.Compose([transforms.Resize((256, 256)),
                                        transforms.CenterCrop(224)])

transformed_image = resize_transform(image)

Transform the image and add the batch dimension to it — everything just like in the previous lessons. What is the first dimension equal to when you call input.shape?

In [ ]:
input = # Your code here
input = # Your code here
input.shape

Now let us load the model.

In [ ]:
model = models.vgg16(weights='IMAGENET1K_V1') # Loading VGG16
model.eval();

Get the output of the model. Which classes are in the top-3?

In [ ]:
output = # Your code here
output = F.softmax(output, dim=1)
prediction_score, pred_label_idx = # Your code here

pred_label_idx.squeeze_().detach().numpy()

predicted_labels = [categories[i] for i in pred_label_idx]
print('Predicted:', predicted_labels, ',\nprobabilities: ', prediction_score.squeeze().detach().numpy(), ')')

Good, in the top-3 our model is confident that we have a dog in front of us, however, if you look at the top 5 you will see that the class tabby (a cat) is also among the model's guesses. Let us move on to the attributions.

**Computing LRP-based attributions.**

First let us apply the straightforward implementation of LRP, where the default epsilon rule is used for every layer. In the hyperparameters of the method there will be nothing new for us, except that there is no need to specify a baseline example. We do not need it by the definition of the method.

In [ ]:
model.zero_grad()

lrp = LRP(model)
basic_attributions_lrp = lrp.attribute(input, target=pred_label_idx[0].item())

Let us visualise the attributions using the `viz` module from `captum`.

In [ ]:
_ = viz.visualize_image_attr_multiple(np.transpose(basic_attributions_lrp.squeeze().cpu().detach().numpy(), (1,2,0)),
                                      np.array(transformed_image),
                                      ["original_image", "heat_map"],
                                      ["all", "all"],
                                      show_colorbar=True,
                                      titles=['Input image', 'Basic LRP map'])

Now let us change the propagation rules for different layers. At the moment of writing this course (2024) Captum has implemented: LRP-Epsilon, LRP-0, LRP-Gamma, LRP-Alpha-Beta and Identity-Rule.

Go back to the trainer and match the rules with the recommendations, according to the theory from the module.

Now let us apply the theory in practice. To begin with, let us apply some rule (to be definite, let us take $epsilon$) to all the layers of the network.

Extract all the layers of VGG16 as a list. How many objects did you get?

In [ ]:
layers = # Your code here
num_layers = len(layers)

print(num_layers)

In [ ]:
# epsilon-rule for each layer

for idx_layer in range(1, num_layers):
  setattr(layers[idx_layer], "rule", EpsilonRule(epsilon=0.1))

lrp = LRP(model)
epsilon_attributions_lrp = lrp.attribute(input,
                                target=pred_label_idx[0].item())

In [ ]:
_ = viz.visualize_image_attr_multiple(np.transpose(epsilon_attributions_lrp.squeeze().cpu().detach().numpy(), (1,2,0)),
                                      np.array(transformed_image),
                                      ["original_image", "heat_map"],
                                      ["all", "all"],
                                      show_colorbar=True,
                                      titles=['Input image', 'Epsilon-rule LRP map'])

The map has become more sparse, which for us means that the map is more informative. The fewer important regions the attribution method highlights, the more pinpointed our explanation is.

Now let us combine several rules according to the recommendations.

In [ ]:
# The basic application of a combination of layers

for idx_layer in range(1, num_layers):
    if idx_layer <= 16:
        setattr(layers[idx_layer], "rule", GammaRule()) # Gamma Rule
    elif 17 <= idx_layer <= 30:
        setattr(layers[idx_layer], "rule", EpsilonRule()) # Epsilon-Rule
    elif idx_layer >= 31:
        setattr(layers[idx_layer], "rule", EpsilonRule(epsilon=0)) # Equivalent to the zero-Rule

lrp = LRP(model)
combined_attributions_lrp = lrp.attribute(input,
                                target=pred_label_idx[0].item())

In [ ]:
_ = viz.visualize_image_attr_multiple(np.transpose(combined_attributions_lrp.squeeze().cpu().detach().numpy(), (1,2,0)),
                                      np.array(transformed_image),
                                      ["original_image", "heat_map"],
                                      ["all", "all"],
                                      show_colorbar=True,
                                      titles=['Input image', 'Combined LRP map'])

At this step we encourage you to experiment! Combine the rules in your own way and share the results in the comments to the special step!

In [ ]:
# The a-b rule is applied to the lower layers of the network (the first ones from the input)

for idx_layer in range(1, num_layers):
  pass


lrp = LRP(model)
custom_attributions_lrp = lrp.attribute(input,
                                target=pred_label_idx[0].item())

In [ ]:
_ = viz.visualize_image_attr_multiple(np.transpose(custom_attributions_lrp.squeeze().cpu().detach().numpy(), (1,2,0)),
                                      np.array(transformed_image),
                                      ["original_image", "heat_map"],
                                      ["all", "all"],
                                      show_colorbar=True,
                                      titles=['Input image', 'Custom LRP map'])

**Evaluating the activation maps.**

Now let us come back to evaluating the maps. Until now we have been evaluating them subjectively — in terms of "I like it and it is clear" or the other way round.

Evaluating the quality of explanation methods is a separate field, which we will only touch upon in passing in the module. But we would like you to leave the course with the understanding that methods can be evaluated not only subjectively!

So let us look at one of the existing metrics — the **Gini index.**

As in other subfields of machine learning, informally it shows the degree of sparsity of the explanation. For us, the more sparse the map is, the more informative it is assumed to be.

By definition:
$$G(v)=1 - 2*∑_{j}\frac{v_j}{||v||_1}*\frac{d-j+0.5}{d}$$

where  \
$v = [v_1, v_2, v_3, .....,v_d], v_i \leq v_i+1$ is the flattened activation map, with the values taken in absolute value and sorted in increasing order. \
$$||v||_p = (\sum_jv_j^p)^{\frac{1}{p}}$$

More about the index [here](https://arxiv.org/pdf/0811.4706v2.pdf).

Let us implement it in code!

In [ ]:
def get_gini_index(attribution_map):

  flattened_map = attribution_map.flatten().detach().numpy()
  flattened_map_abs_and_sorted = sorted(np.abs(flattened_map)) #We sort in increasing order

  d = len(flattened_map_abs_and_sorted) #The length of the vector
  vector_norm = np.linalg.norm(flattened_map_abs_and_sorted, 1)

  s = 0
  for i in range(1, d+1): #We get the sums

    coef = (d-i+0.5)/d
    first = flattened_map_abs_and_sorted[i-1]/vector_norm
    s += coef*first


  gini_index = 1-2*s

  return gini_index

Evaluate each map with the Gini index. Which map is the most sparse? (Not counting your own map)

In [ ]:
carts = # Your code here
cart_codes = ['basic', 'epsilon', 'combined']

for card, cart_code in zip(carts, cart_codes):

  print(cart_code + f' card. Gini index = {get_gini_index(card)}')

**That is all! Thank you!** 😊

Note: in the notebook we use the VGG16 model, since the default rules for LRP are currently set up not for all models. So, for example, for ResNet they can be problematic to apply, and it is better to check the applicability to the architectures in the [documentation](https://captum.ai/docs/introduction).

**Material for further study: user implementations of LRP:**
1. An implementation of LRP for pyTorch can also be found here: https://github.com/fhvilshoj/TorchLRP (try it out in practice yourself and write a review)

2. Here there is a user implementation [with interactivity](https://github.com/kaifishr/PyTorchRelevancePropagation/tree/master)

3. [Here](https://git.tu-berlin.de/gmontavon/lrp-tutorial) is an implementation of the method from scratch